# PANDAPROSUMER EXAMPLE: SIMPLE HEAT STORAGE (TWO MODES)

## DESCRIPTION
This tutorial illustrates the **Simple Heat Storage** controller, which can be used in two ways:

1. **GenericMapping (power-only)**: The storage receives and delivers power only (`q_received_kw` in; `soc`, `q_delivered_kw` out). Suitable when upstream/downstream elements work with power only.

2. **FluidMixMapping (uniform tank)**: The storage is modelled as a uniform-temperature tank with temperature and mass-flow in/out. Use when connecting to fluid-based elements (e.g. heat pump, heat demand with fluid). Optional `min_temp_c` and `max_temp_c` on the element allow SOC to be derived from tank temperature.

Time series data is defined in the notebook (no external files). After each part we run the timeseries and plot SOC and delivered power.

## Glossary
- **Network**: A configuration of connected energy generators and consumers.
- **Element**: A single generator or consumer.
- **Controller**: The logic that defines an element's behaviour.
- **Prosumer**: Container holding elements and their controllers.
- **Const Profile Controller**: Distributes time-dependent input data to element controllers.
- **Mapping**: Connection between two controllers (GenericMapping for power/data; FluidMixMapping for temperature and mass flow).

---
# Part 1: Power-only mode (GenericMapping)

Chain: **Const profile (supply power)** → **Heat storage** → **Heat demand**.

The const profile provides a supply power and demand data; the storage receives power and delivers to the demand.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandapower.timeseries.data_sources.frame_data import DFData

current_directory = os.getcwd()
parent_directory = os.path.dirname(current_directory)
sys.path.insert(0, parent_directory)

In [ ]:
# Time range and resolution
start = '2020-01-01 00:00:00'
end = '2020-01-01 23:59:59'
time_resolution_s = 900
dur = pd.date_range(start=start, end=end, freq=f'{time_resolution_s}s', tz='utc')
n_steps = len(dur)

In [ ]:
# Part 1 input: supply power (to storage) and demand (for heat demand element)
supply_kw = np.zeros(n_steps)
supply_kw[0:8] = 25.0   # charge 25 kW for first 2 h
supply_kw[20:24] = 20.0 # charge again later
demand_kw = np.zeros(n_steps)
demand_kw[4:8] = 60.0   # demand spike 60 kW for 4 steps
demand_kw[12:16] = 25.0
df1 = pd.DataFrame({
    'supply_power': supply_kw,
    'demand_power': demand_kw,
    't_feed_demand_c': 80.0,
    't_return_demand_c': 20.0
}, index=dur)
profile1 = DFData(df1)
df1.head(10)

In [ ]:
from pandaprosumer.create import create_empty_prosumer_container, create_period
from pandaprosumer.create_controlled import (
    create_controlled_const_profile,
    create_controlled_heat_storage,
    create_controlled_heat_demand
)

prosumer1 = create_empty_prosumer_container()
period_id = create_period(prosumer1, time_resolution_s, start, end, 'utc', 'default')

cp_input = ['supply_power', 'demand_power', 't_feed_demand_c', 't_return_demand_c']
cp_result = ['supply_power', 'qdemand_kw', 't_feed_demand_c', 't_return_demand_c']
cp_idx = create_controlled_const_profile(prosumer1, cp_input, cp_result, profile1, period_id, 0, 0)

hs_idx = create_controlled_heat_storage(prosumer1, q_capacity_kwh=100.0, name='tank_power_only',
                                        period=period_id, level=1, order=0)
hd_idx = create_controlled_heat_demand(prosumer1, period=period_id, level=1, order=1, name='heat_consumer')

In [ ]:
from pandaprosumer.mapping import GenericMapping

# Const profile -> Heat storage: supply power as q_received_kw
GenericMapping(prosumer1, initiator_id=cp_idx, initiator_column='supply_power',
               responder_id=hs_idx, responder_column='q_received_kw', order=0)
# Const profile -> Heat demand: demand request and temperatures
GenericMapping(prosumer1, initiator_id=cp_idx,
               initiator_column=['qdemand_kw', 't_feed_demand_c', 't_return_demand_c'],
               responder_id=hd_idx, responder_column=['q_demand_kw', 't_feed_demand_c', 't_return_demand_c'], order=1)
# Heat storage -> Heat demand: delivered power as q_received_kw
GenericMapping(prosumer1, initiator_id=hs_idx, initiator_column='q_delivered_kw',
               responder_id=hd_idx, responder_column='q_received_kw', order=0)

In [ ]:
from pandaprosumer.run_time_series import run_timeseries

run_timeseries(prosumer1, period_id, verbose=False)

In [ ]:
res1 = prosumer1.time_series.copy()
# time_series rows are indexed by integer; use name column to get the storage results
res1_idx = res1[res1['name'] == 'tank_power_only'].index[0]
df_hs1 = res1.data_source.loc[res1_idx].df

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
df_hs1.soc.plot(ax=axes[0], color='C0')
axes[0].set_ylabel('SOC (-)')
axes[0].set_title('Part 1 (GenericMapping): Heat storage SOC')
axes[0].grid(True, alpha=0.3)
df_hs1.q_delivered_kw.plot(ax=axes[1], color='C1')
axes[1].set_ylabel('Power (kW)')
axes[1].set_title('Delivered power')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# Part 2: FluidMixMapping (uniform tank)

Chain: **Const profile** → **Electric boiler** → **Heat storage (uniform tank)** → **Heat demand**.

The electric boiler supplies the storage with fluid (temperature + mass flow); the storage is configured with `capacity_kg`, optional `init_temperature_c`, `min_temp_c`, `max_temp_c` for SOC from temperature. The run uses `continue_on_divergence=True` so that if the control loop does not converge in some timesteps, the time series still completes (you may see gaps or zeros in results for those steps).

In [ ]:
# Part 2 input: demand power and feed/return temperatures for the heat demand (same time range as Part 1)
demand_kw2 = np.zeros(n_steps)
demand_kw2[6:10] = 60.0
demand_kw2[14:18] = 30.0
df2 = pd.DataFrame({
    'demand_power': demand_kw2,
    't_feed_demand_c': 55.0,
    't_return_demand_c': 25.0
}, index=dur)
profile2 = DFData(df2)
df2.head(10)

In [ ]:
from pandaprosumer.create_controlled import create_controlled_electric_boiler

prosumer2 = create_empty_prosumer_container()
period_id2 = create_period(prosumer2, time_resolution_s, start, end, 'utc', 'default')

cp_input2 = ['demand_power', 't_feed_demand_c', 't_return_demand_c']
cp_result2 = ['qdemand_kw', 't_feed_demand_c', 't_return_demand_c']
cp_idx2 = create_controlled_const_profile(prosumer2, cp_input2, cp_result2, profile2, period_id2, 0, 0)

eb_idx = create_controlled_electric_boiler(prosumer2, max_p_kw=150.0, name='electric_boiler',
                                           period=period_id2, level=1, order=0)

capacity_kg = 2000.0
init_t = 45.0
min_t, max_t = 30.0, 70.0
hs_idx2 = create_controlled_heat_storage(prosumer2, q_capacity_kwh=50.0, name='tank_fluid_mix',
                                         capacity_kg=capacity_kg, init_temperature_c=init_t,
                                         init_temperature=init_t, min_temp_c=min_t, max_temp_c=max_t,
                                         period=period_id2, level=1, order=1)
hd_idx2 = create_controlled_heat_demand(prosumer2, period=period_id2, level=1, order=2, name='heat_consumer')

In [ ]:
from pandaprosumer.mapping import GenericMapping, FluidMixMapping

GenericMapping(prosumer2, initiator_id=cp_idx2,
               initiator_column=['qdemand_kw', 't_feed_demand_c', 't_return_demand_c'],
               responder_id=hd_idx2, responder_column=['q_demand_kw', 't_feed_demand_c', 't_return_demand_c'], order=0)
FluidMixMapping(prosumer2, initiator_id=eb_idx, responder_id=hs_idx2, order=0)
FluidMixMapping(prosumer2, initiator_id=hs_idx2, responder_id=hd_idx2, order=0)

In [ ]:
run_timeseries(prosumer2, period_id2, verbose=False, max_iter=15, continue_on_divergence=True)

In [ ]:
res2 = prosumer2.time_series.copy()
res2_idx = res2[res2['name'] == 'tank_fluid_mix'].index[0]
df_hs2 = res2.data_source.loc[res2_idx].df

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
df_hs2.soc.plot(ax=axes[0], color='C0')
axes[0].set_ylabel('SOC (-)')
axes[0].set_title('Part 2 (FluidMixMapping): Heat storage SOC (from tank temperature)')
axes[0].grid(True, alpha=0.3)
df_hs2.q_delivered_kw.plot(ax=axes[1], color='C1')
axes[1].set_ylabel('Power (kW)')
axes[1].set_title('Delivered power')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary
- **Part 1**: Power-only chain with GenericMapping; storage input is `q_received_kw`, output is `soc` and `q_delivered_kw`. Fully convergent.
- **Part 2**: Fluid chain (Electric boiler → Uniform heat storage → Heat demand) with FluidMixMapping; storage uses uniform tank and optional SOC from temperature (`min_temp_c`, `max_temp_c`). Run uses `continue_on_divergence=True` so the time series completes even if the control loop does not converge in some steps; same result columns `soc`, `q_delivered_kw`.